<a href="https://colab.research.google.com/github/siinwook/Deep-Learning-from-Scratch/blob/main/ch04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install import_ipynb
import sys, import_ipynb

from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/Colab Notebooks/Deep-Learning-from-Scratch 1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.3 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Colab Notebooks/Deep-Learning-from-Scratch 1


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sum_square_error

In [ ]:
def sum_square_error(y,t): # y: f(x), t: target
  return np.sum((y-t)**2)

cross_entropy_error (one-hot label)

In [ ]:
def cross_entropy_error(y,t):
  y=y.reshape(1,y.size)
  t=t.reshape(1,t.size)

  batch_size = y.shape[0]
  delta = 1e-7
  return -np.sum(t*np.log(y+delta)) / batch_size

cross_entropy_error (numerical label)

In [ ]:
def cross_entropy_error(y,t):
  y=y.reshape(1,y.size)
  t=t.reshape(1,t.size)

  batch_size = y.shape[0]
  delta = 1e-7
  return -np.sum(np.log(y[np.arange(batch_size), t] + delta)) / batch_size

numerical gradient (n dim)

In [ ]:
def numerical_gradient(f, x):
    h = 1e-4 # 0.0001
    grad = np.zeros_like(x)

    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        tmp_val = x[idx]
        x[idx] = float(tmp_val) + h
        fxh1 = f(x) # f(x+h)

        x[idx] = tmp_val - h
        fxh2 = f(x) # f(x-h)
        grad[idx] = (fxh1 - fxh2) / (2*h)

        x[idx] = tmp_val # recover
        it.iternext()

    return grad

2-layer network

In [ ]:
from functions import sigmoid, softmax

class TwoLayerNet:
  def __init__(self, input_size, hidden_size, output_size, weight_init_std=0.01):
    self.params = {}
    self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size)
    self.params['b1'] = weight_init_std * np.random.randn(hidden_size)
    self.params['W2'] = weight_init_std * np.random.randn(hidden_size, output_size)
    self.params['b2'] = weight_init_std * np.random.randn(output_size)

  def predict(self,x):
    W1, W2=self.params['W1'], self.params['W2']
    b1,b2 = self.params['b1'], self.params['b2']

    z1 = sigmoid(np.dot(x,W1) + b1) #Layer 1
    y = softmax(np.dot(z1,W2) + b2) #Layer 2

    return y

  def loss(self, x, t):
    y = self.predict(x)
    return cross_entropy_error(y,t)

  def accuracy(self,x,t):
    y = self.predict(x)
    y = np.argmax(y, axis=1)
    t = np.argmax(t, axis=1)

    accuracy = np.sum((y==t).astype(int)) / float(x.shape[0])
    return accuracy

  def numerical_gradient(self, x, t):
    loss_W = lambda W: self.loss(x,t)

    grads={}
    grads['W1'] = numerical_gradient(loss_W, self.params['W1'])
    grads['b1'] = numerical_gradient(loss_W, self.params['b1'])
    grads['W2'] = numerical_gradient(loss_W, self.params['W2'])
    grads['b2'] = numerical_gradient(loss_W, self.params['b2'])

    return grads


mini batch training

In [ ]:
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import tqdm

(x_train, t_train), (x_test, t_test) = mnist.load_data()
x_train = x_train.reshape(60000,784)
x_test = x_test.reshape(10000,784)
x_train = x_train/255.0
x_test = x_test/255.0
t_train = to_categorical(t_train, 10).astype(int) # one-hot encoding
t_test = to_categorical(t_test, 10).astype(int)

train_loss_list = []
train_acc_list = []
test_acc_list = []

#hyper parameter
iters_num = 10000
train_size = x_train.shape[0] # 60000
batch_size = 100
learning_rate = 0.1

network = TwoLayerNet(input_size = 784, hidden_size = 50, output_size=10)

iter_per_epoch = max(train_size / batch_size, 1) # ex) 60000 / 100 = 600

for i in tqdm.tqdm(range(iters_num)):
  print(i)
  batch_mask = np.random.choice(train_size,batch_size)
  x_batch = x_train[batch_mask]
  t_batch = t_train[batch_mask]

  grad = network.numerical_gradient(x_batch, t_batch)

  for key in ('W1', 'b1', 'W2', 'b2'):
    network.params[key] -= learning_rate * grad[key]

  loss = network.loss(x_batch, t_batch)
  train_loss_list.append(loss)

  # accuracy per epoch
  if i % iter_per_epoch == 0: # ex) each 600 iters
    train_acc = network.accuracy(x_train, t_train)
    test_acc = network.accuracy(x_test, t_test)
    train_acc_list.append(train_acc)
    test_acc_list.append(test_acc)
    print("train acc, test acc | " + str(train_acc) + ", " + str(test_acc))